In [1]:
import sys
sys.path.append('../')

import numpy as np
import scqubits.settings as settings
settings.OVERLAP_THRESHOLD = 0.3
import utils_2Q_gate_zp as ut
import qutip as qt
import pandas as pd
import scqubits as scq
import qutip as qt
import numpy as np
from matplotlib import pyplot as plt
from qutip.qip.operations import rz, cz_gate, cnot, rx, hadamard_transform, swap
import cmath
import scipy.sparse as ssp
from sympy import symbols
from joblib import Parallel, delayed
from multiprocessing import Pool
from IPython.display import display, Math

In [2]:
drive_phi, drive_theta, truncation = False, True, 10
ncut, phi_cut = 30, 100
# Define system parameters (in GHz)
EL = 0.377  # Inductive energy
EJ = 6.013  # Josephson energy
EC_phi = 1.142  # Phi mode charging energy
EC_theta = 0.092  # Theta mode charging energy

# Compute derived parameters
E_CJ = 2 * EC_phi
E_C = 2 / (1 / EC_theta - 1 / EC_phi)

# Create the grid for the phi coordinate
phi_grid = scq.Grid1d(-6 * np.pi, 6 * np.pi, phi_cut)

# Initialize the Zero-Pi qubit system
zero_pi = scq.ZeroPi(
    grid=phi_grid,
    EJ=EJ,
    EL=EL,
    ECJ=E_CJ,
    EC=E_C,
    dEJ=0.0,
    ng=0.8,
    flux=0.7,
    ncut=ncut,
    truncated_dim=truncation,
)

# Compute matrix elements for the theta and phi operators
n_Theta = zero_pi.matrixelement_table(operator="n_theta_operator", evals_count=truncation)
n_Phi = zero_pi.matrixelement_table(operator="i_d_dphi_operator", evals_count=truncation)
# Compute the eigenvalues and construct the Hamiltonian
evals, ekets = zero_pi.eigensys(evals_count=truncation)

print('imag(eket0)=0 -->', np.all(np.imag(np.array(ekets))==0))
print('imag(n_Theta)=0 -->', np.all(np.imag(n_Theta)==0))

imag(eket0)=0 --> True
imag(n_Theta)=0 --> True


### Import check

In [2]:
truc_import = 500
folder = f'../../data/3ncut_two_zeropi/truc1={truc_import}/'
eval0 = pd.read_csv(folder+ 'eval0.txt').to_numpy().flatten()
eval1 = pd.read_csv(folder+ 'eval1.txt').to_numpy().flatten()
eket0 = np.load(folder+'eket0.npy')
eket1 = np.load(folder+'eket1.npy')
n_theta0 = np.load(folder+'n_theta0.npy')
n_theta1 = np.load(folder+'n_theta1.npy')
print('imag(eket0)=0 -->', np.all(np.imag(eket0)==0))
print('imag(n_theta0)=0 -->', np.all(np.imag(n_theta0)==0))

imag(eket0)=0 --> True
imag(n_theta0)=0 --> True


In [3]:
truc1, truc_tot, charge_pick = 300, 1000, True
folder = f'../../data/3ncut_two_zeropi/truc1={truc1}_truc2={truc_tot}_pick={charge_pick}/'
hspace_true = pd.read_csv(folder+ 'hspace_full.txt').to_numpy().flatten().tolist()
eval_true = 2*np.pi* pd.read_csv(folder+ 'eval_tot.txt').to_numpy().flatten()
n_theta0_true = 2*np.pi* np.load(folder+'n_theta0_dress.npy')
n_theta1_true = 2*np.pi* np.load(folder+'n_theta1_dress.npy')
eket_true = ssp.csr_matrix(np.load(folder+ 'eket_tot.npy'))

In [4]:
truc1, truc_tot, charge_pick = 300, 1000, False
folder = f'../../data/3ncut_two_zeropi/truc1={truc1}_truc2={truc_tot}_pick={charge_pick}/'
hspace_false = pd.read_csv(folder+ 'hspace_full.txt').to_numpy().flatten().tolist()
eval_false = 2*np.pi* np.abs(pd.read_csv(folder+ 'eval_tot.txt').to_numpy().flatten())
n_theta0_false = 2*np.pi* np.load(folder+'n_theta0_dress.npy')
n_theta1_false = 2*np.pi* np.load(folder+'n_theta1_dress.npy')
eket_false = ssp.csr_matrix(np.load(folder+ 'eket_tot.npy'))

print('imag(eket_true)=0 -->', np.all(np.imag(eket_true.todense())==0))
print('imag(n_theta0_true)=0 -->', np.all(np.imag(n_theta0_true)==0))
print('imag(eket_false)=0 -->', np.all(np.imag(eket_false.todense())==0))
print('imag(n_theta0_false)=0 -->', np.all(np.imag(n_theta0_false)==0))

imag(eket_true)=0 --> False
imag(n_theta0_true)=0 --> False
imag(eket_false)=0 --> False
imag(n_theta0_false)=0 --> False


In [5]:
=

SyntaxError: invalid syntax (1763773627.py, line 1)

In [ ]:
index_500 = hspace_true.index('30-2') # 497
index_end = hspace_true.index('77-0') # 999
print( 'index_500=', index_500)
print( 'index_end=', index_end)
index_true = np.arange(index_end+1)
eval_true_part = eval_true[:index_end+1]
hspace_true_part = hspace_true[:index_end+1]
n_theta0_true_part = ut.truncate_2(n_theta0_true, index_true)
n_theta1_true_part = ut.truncate_2(n_theta1_true, index_true)

index_500= 190
index_end= 352


In [ ]:
index_false = [hspace_false.index(i) for i in hspace_true[:index_end+1]]
eval_false_part = np.abs(ut.truncate_2(eval_false, index_false).full().flatten())
hspace_false_part = np.array(hspace_false)[index_false].tolist()
n_theta0_false_part = ut.truncate_2(n_theta0_false, index_false)
n_theta1_false_part = ut.truncate_2(n_theta1_false, index_false)

In [ ]:
# print(eval_true_part)
# print(eval_false_part)
print(np.allclose(eval_true_part, eval_false_part, atol=1e-8))
# print(hspace_true_part)
# print(hspace_false_part)
print(hspace_false_part==hspace_false_part)
print(np.allclose(n_theta0_true_part, qt.Qobj(np.abs(n_theta0_false_part.full())), atol=1e-7))
print(np.allclose(n_theta1_true_part, qt.Qobj(np.abs(n_theta1_false_part.full())), atol=1e-7))


True
True
True
True


In [ ]:
# np.save(folder+'n_theta0_dress.npy', n_theta0_false)
# np.save(folder+'n_theta1_dress.npy', n_theta1_false)

### check eket of single zeropi

In [ ]:
Ec0=1.0
truc1=10
truc_tot=50
charge_pick=False
n_cut=20
phi_cut=30

zp = scq.Circuit(ut.zp_yml, from_file=False)
zp.Ec0 = Ec0
zp.configure(transformation_matrix=np.linalg.inv(ut.transform_2zeropi))

##############################################################################################
### Construct subsystem, calculate eigenvalues
system_hierarchy = [[1,2],  [5,6]]
subsystem_trunc_dims = [10, 10]
zp.configure(system_hierarchy=system_hierarchy,
            subsystem_trunc_dims=subsystem_trunc_dims)

zp.cutoff_ext_1, zp.cutoff_ext_5 = phi_cut, phi_cut
zp.cutoff_n_2, zp.cutoff_n_6 = n_cut, n_cut

### the two-line code below takes time when truc1 is large
eval0, eket0 = zp.subsystems[0].eigensys(evals_count=truc1)
eval1, eket1 = zp.subsystems[1].eigensys(evals_count=truc1)

sorted_idx0 = np.argsort(eval0)
eval0 = eval0[sorted_idx0]
eval0 = eval0 - eval0[0]
eket0 = ssp.csr_matrix([eket0[:,idx] for idx in range(truc1)])

sorted_idx1 = np.argsort(eval1)
eval1 = eval1[sorted_idx1]
eval1 = eval1 - eval1[0]
eket1 = ssp.csr_matrix([eket1[:,idx] for idx in range(truc1)])

# get the n-operator in qubit basis of single qubit
n_theta0 = (eket0 @ zp.subsystems[0].n2_operator() @ eket0.conj().T).todense()
n_theta1 = (eket1 @ zp.subsystems[1].n6_operator() @ eket1.conj().T).todense()

print(np.all(np.imag(eket0.toarray())==0))
print(np.all(np.imag(n_theta0)==0))

True
True
